In [4]:
!pip install pyspark -q

In [5]:
!pip install boto3
!pip install pyarrow

In [6]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

Mounted at /content/drive
✅ Drive mounted successfully


In [7]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [8]:

import requests
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import boto3, pyarrow as pa, pyarrow.parquet as pq
from io import BytesIO

response = requests.get("https://api.data.gov.my/weather/warning/earthquake")
#data = response.json()

spark = SparkSession.builder \
    .appName("MyEarthQuakeApp") \
    .getOrCreate()

# Test it
spark.version

# Wrap in a list — critical! Passing bare text iterates characters instead
rdd = spark.sparkContext.parallelize([response.text])
df = spark.read.json(rdd)

df.printSchema()
#df.show()
df = df.withColumn(
    "utcdatetime_ts",
    F.expr("try_to_timestamp(utcdatetime, \"yyyy-MM-dd'T'HH:mm:ss\")")
)
df.orderBy(F.desc("utcdatetime_ts")).show(10)

pdf = df.toPandas()
buffer = BytesIO()
pq.write_table(pa.Table.from_pandas(pdf), buffer)
buffer.seek(0)

s3 = boto3.client("s3",
    endpoint_url=G_MINIO_ENDPOINT,
    aws_access_key_id=G_MINIO_ACCESS_KEY,
    aws_secret_access_key=G_MINIO_SECRET_KEY
)
s3.put_object(Bucket="rawdatasets", Key="earthquake/earthquake_data.parquet", Body=buffer.getvalue())

root
 |-- depth: double (nullable = true)
 |-- lat: double (nullable = true)
 |-- lat_vector: string (nullable = true)
 |-- localdatetime: string (nullable = true)
 |-- location: string (nullable = true)
 |-- location_original: string (nullable = true)
 |-- lon: double (nullable = true)
 |-- lon_vector: string (nullable = true)
 |-- magdefault: double (nullable = true)
 |-- magtypedefault: string (nullable = true)
 |-- n_distancemas: string (nullable = true)
 |-- n_distancerest: string (nullable = true)
 |-- nbm_distancemas: string (nullable = true)
 |-- nbm_distancerest: string (nullable = true)
 |-- status: string (nullable = true)
 |-- utcdatetime: string (nullable = true)
 |-- visible: boolean (nullable = true)

+-----+---------+----------+-------------------+--------------------+--------------------+-----------+-----------+----------+--------------+--------------------+--------------------+--------------------+--------------------+------+-------------------+-------+---------------

{'ResponseMetadata': {'RequestId': '18980F16E20BCED2',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '0',
   'date': 'Fri, 27 Feb 2026 09:04:05 GMT',
   'etag': '"107edc542668e3b659c1662e8422d70f"',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'via': '1.1 Caddy',
   'x-amz-checksum-crc32': 'qM71fQ==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '18980F16E20BCED2',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '6632',
   'x-ratelimit-remaining': '6632',
   'x-xss-protection': '1; mode=block'},
  'RetryAttempts': 0},
 'ETag': '"107edc542668e3b659c1662e8422d70f"',
 'ChecksumCRC32': 'qM71fQ==',
 'ChecksumType': 'FULL_OBJECT'}

In [9]:
spark.stop()